# Mini-TP 4 — Puntúa tu modelo sobre un flujo (starter)

**Operaciones de Aprendizaje Automático II · CEIA – FIUBA · Entrega individual, esta semana**

Aplica **tu modelo** (el de la Sesión 1) sobre un **flujo de eventos** en lugar de pedidos sueltos. Las celdas de la actividad fueron completadas en esta versión.

**Qué debes entregar**
1. Un **flujo** de eventos con las *features* de tu modelo (simulado, o consumido de Kafka).
2. Un **consumidor** que puntúe tu modelo sobre **cada** evento (inferencia online).
3. **Métricas por ventana:** throughput (ev/s), latencia **p95** y un indicador simple de **drift** de entradas.
4. Una **alerta** cuando una métrica cruce un umbral.
5. Una **comparación** contra puntuar el mismo lote en **batch**, con una breve reflexión.

**Se evalúa:** que corra de punta a punta; inferencia online sobre el flujo; métricas por ventana correctas; y la reflexión streaming vs batch.

> Apóyate en `streaming_tutorial.ipynb`. Entorno uv: `uv add scikit-learn joblib numpy kafka-python`.


In [1]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn", "joblib", "numpy", "pandas"], check=True)
print("Dependencias listas.")


Dependencias listas.


## 1. Tu modelo

Carga **tu** modelo. Si aún no lo tienes a mano, el placeholder deja probar la mecánica.

In [2]:
from pathlib import Path
import sys, joblib, json, numpy as np, pandas as pd

# Busca common/ tanto si el notebook se ejecuta desde el repo raíz como desde mini_tp4_/.
def encontrar_common():
    candidatos = [Path.cwd()/"common", Path.cwd().parent/"common", Path("/mnt/data/common")]
    for p in candidatos:
        if (p/"artifacts"/"model.pkl").exists() and (p/"artifacts"/"data_dict.json").exists():
            return p.resolve()
    raise FileNotFoundError("No encontré common/artifacts/model.pkl y data_dict.json")

COMMON = encontrar_common()
if str(COMMON.parent) not in sys.path:
    sys.path.insert(0, str(COMMON.parent))

from common.inference import load_model, FEATURE_FIELDS

modelo = load_model()  # StrokeModel: modelo + preprocesamiento del proyecto
DATASET = COMMON / "data" / "healthcare-dataset-stroke-data.csv"
df_fuente = pd.read_csv(DATASET)
df_fuente["bmi"] = df_fuente["bmi"].fillna(df_fuente["bmi"].median())

print(f"Modelo cargado: {modelo.name} v{modelo.version}")
print("Tipo:", type(modelo.model).__name__)
print("Features crudas:", FEATURE_FIELDS)
print("Métricas originales:", modelo.metrics)


Modelo cargado: stroke_prediction_model_prod v1
Tipo: RandomForestClassifier
Features crudas: ['gender', 'age', 'hypertension', 'heart_disease', 'ever_married', 'work_type', 'Residence_type', 'avg_glucose_level', 'bmi', 'smoking_status']
Métricas originales: {'accuracy': 0.9660668380462725, 'precision': 0.9758403361344538, 'recall': 0.9557613168724279, 'f1_score': 0.9656964656964657, 'train_observations': 7777, 'test_observations': 1945}


## 2. El flujo

Simula un flujo con las features de tu modelo. Introduce un cambio de distribución a mitad de camino para poder detectar drift. (Alternativa: consumir de Kafka como en el tutorial.)

In [3]:
import queue, threading, time, random

stream = queue.Queue()
STOP = object()
N = 800
rng_stream = random.Random(42)
eventos_generados = []

# Tomamos eventos reales del dataset. En la segunda mitad introducimos drift
# aumentando edad y glucosa para modificar la distribución de entrada sin inventar
# un dominio distinto al del modelo de stroke.
registros = df_fuente[FEATURE_FIELDS].to_dict(orient="records")

def productor():
    for i in range(N):
        ev = dict(registros[rng_stream.randrange(len(registros))])
        if i >= N // 2:
            ev["age"] = min(82.0, float(ev["age"]) + 15.0)
            ev["avg_glucose_level"] = float(ev["avg_glucose_level"]) + 35.0
        item = {"features": ev, "t": time.time()}
        eventos_generados.append({"features": dict(ev), "t": item["t"]})
        stream.put(item)
        time.sleep(0.0005)
    stream.put(STOP)

print("Productor listo con eventos del dominio stroke.")


Productor listo con eventos del dominio stroke.


## 3. Consumidor con inferencia online + métricas por ventana

In [4]:
from collections import deque
import numpy as np

latencias = []
ventana = deque()
alertas = []
metricas_ventana = []

BASELINE_AGE = float(df_fuente["age"].mean())
BASELINE_AGE_STD = float(df_fuente["age"].std())
UMBRAL_DRIFT = 0.45  # desplazamiento de la media de edad medido en desvíos estándar


def consumidor():
    procesados = 0
    while True:
        ev = stream.get()
        if ev is STOP:
            break

        t0_pred = time.perf_counter()
        pred, label, proba = modelo.predict_one(ev["features"])
        lat_ms = (time.perf_counter() - t0_pred) * 1000
        latencias.append(lat_ms)

        ahora = ev["t"]
        ventana.append((ahora, float(ev["features"]["age"]), pred, lat_ms))
        while ventana and ahora - ventana[0][0] > 1.0:
            ventana.popleft()

        procesados += 1
        if procesados % 200 == 0:
            ages = np.array([w[1] for w in ventana], dtype=float)
            lats = np.array([w[3] for w in ventana], dtype=float)
            media_age = float(ages.mean())
            drift_score = abs(media_age - BASELINE_AGE) / max(BASELINE_AGE_STD, 1e-9)

            if len(ventana) > 1:
                ancho = max(ventana[-1][0] - ventana[0][0], 1e-9)
                throughput_ventana = (len(ventana) - 1) / ancho
            else:
                throughput_ventana = 0.0

            metrica = {
                "procesados": procesados,
                "eventos_ventana": len(ventana),
                "throughput_ev_s": throughput_ventana,
                "p95_ms": float(np.percentile(lats, 95)),
                "media_age": media_age,
                "drift_score": drift_score,
            }
            metricas_ventana.append(metrica)

            if drift_score > UMBRAL_DRIFT:
                alertas.append((procesados, drift_score, media_age))


t0 = time.time()
pt = threading.Thread(target=productor)
ct = threading.Thread(target=consumidor)
pt.start(); ct.start(); pt.join(); ct.join()
dur = time.time() - t0

print(f"throughput global : {len(latencias)/dur:.0f} ev/s")
print(f"p95 global        : {np.percentile(latencias, 95):.3f} ms")
print("\nMétricas por ventana:")
for m in metricas_ventana:
    print(
        f"  n={m['procesados']:3d} | ventana={m['eventos_ventana']:3d} | "
        f"throughput={m['throughput_ev_s']:.0f} ev/s | p95={m['p95_ms']:.3f} ms | "
        f"age_mean={m['media_age']:.2f} | drift={m['drift_score']:.2f}"
    )
print("\nAlertas de drift:", alertas)


throughput global : 54 ev/s
p95 global        : 24.350 ms

Métricas por ventana:
  n=200 | ventana=200 | throughput=698 ev/s | p95=19.236 ms | age_mean=42.70 | drift=0.02
  n=400 | ventana=400 | throughput=720 ev/s | p95=19.654 ms | age_mean=43.41 | drift=0.01
  n=600 | ventana=600 | throughput=716 ev/s | p95=24.008 ms | age_mean=48.27 | drift=0.22
  n=800 | ventana=724 | throughput=723 ev/s | p95=24.356 ms | age_mean=50.57 | drift=0.32

Alertas de drift: []


## 4. Comparación contra batch

Puntúa el mismo conjunto de eventos de una sola vez (batch) y compara tiempo/uso frente al streaming.

In [5]:
# Mismos 800 eventos, ahora puntuados juntos con predict_batch del wrapper compartido.
LOTE = [ev["features"] for ev in eventos_generados]

t0_batch = time.perf_counter()
batch_result = modelo.predict_batch(LOTE)
batch_total_ms = (time.perf_counter() - t0_batch) * 1000
batch_por_evento_ms = batch_total_ms / len(LOTE)

stream_promedio_ms = float(np.mean(latencias))
stream_p95_ms = float(np.percentile(latencias, 95))
batch_pred = np.array([int(r[0]) for r in batch_result])

print(f"Eventos                     : {len(LOTE)}")
print(f"Batch total                 : {batch_total_ms:.3f} ms")
print(f"Batch por evento (promedio) : {batch_por_evento_ms:.6f} ms")
print(f"Streaming por evento (mean) : {stream_promedio_ms:.6f} ms")
print(f"Streaming por evento (p95)  : {stream_p95_ms:.6f} ms")
print(f"Predicciones positivas batch: {int(batch_pred.sum())}")


Eventos                     : 800
Batch total                 : 39.017 ms
Batch por evento (promedio) : 0.048771 ms
Streaming por evento (mean) : 18.470577 ms
Streaming por evento (p95)  : 24.349520 ms
Predicciones positivas batch: 6


### Reflexión

El streaming permite obtener una predicción apenas llega cada evento y, al mismo tiempo, observar métricas operativas y cambios de distribución. En este caso medí el drift como el desplazamiento de la media de `age` respecto de la distribución original, expresado en desvíos estándar, y generé una alerta al superar el umbral. El batch es más eficiente cuando ya tengo un conjunto completo y no necesito respuesta inmediata, porque aprovecha mejor la inferencia vectorizada. Ante una alerta sostenida primero validaría calidad de datos y luego evaluaría recalibrar o reentrenar el modelo con datos recientes antes de promover una nueva versión.


## 5. (Opcional) Con Kafka en Docker

Levanta un broker y reemplaza la cola por un topic real:
```bash
docker run -d --name redpanda -p 9092:9092 \
  redpandadata/redpanda redpanda start --overprovisioned --smp 1 --check=false
uv add kafka-python
```
Produce a un topic `eventos` y consume puntuando tu modelo (ver la parte 6 del tutorial).